# Chapter 11 — Compression Is Loss

## Question

**If important information cannot be deleted, what exactly must survive when we represent it more compactly?**

Falsifiable structure: if a deterministic naive compactor drops probe items that a typed compactor keeps, survival-by-category separates them at matched budgets. No LLM summariser is called; every transformation below is a fixed string rule, so losses are attributable, not sampled.

## Setup — a frozen source record as ground truth

The evaluator retains this source no matter what any policy would discard. Token counts are fixtures.

In [ ]:
SOURCE = {
    'prohibition': 'Never modify production migrations without explicit approval.',
    'version': '18.2',
    'migration_id': '01947',
    'decision': 'PostgreSQL',
    'rationale': 'event-store workload needs serialisable transactions',
    'rejected': 'Redis (loses transactions)',
    'hypothesis': 'STATUS=suspected: serializer corrupts timestamps',
    'fact': 'seed 01947 reproduces the flake',
    'provenance': 'SOURCE=ADR-007',
    'evidence': 'EVIDENCE: 400-line perf log; finding: p99 doubles under batch writes',
    'transient': 'ran ls twice; shell was slow',
}
SOURCE_TOKENS = 1200
PIN_SPANS = ['Never modify production migrations without explicit approval.', '18.2', '01947']
print(f'{len(SOURCE)} information classes frozen; source {SOURCE_TOKENS} fixture tokens.')

## Baseline — RAW keeps everything, costs everything

In [ ]:
RAW = dict(SOURCE)
RAW_TOKENS = SOURCE_TOKENS
print(f'RAW: {RAW_TOKENS} tokens; survival questions do not arise.')

## Intervention — four deterministic compactions at matched budgets

NAIVE shortens, flattens status, drops provenance. TYPED honours Chapter 7 contracts: PIN spans copied verbatim, decisions keep rationale, evidence becomes identity-plus-finding, uncertainty status retained, transient detail omitted. ORACLE uses probe knowledge and is a ceiling, never a proposal.

In [ ]:
def naive_compact(src):
    return {
        'prohibition': 'Be careful with migrations.',
        'version': '18',
        'migration_id': '1947',
        'decision': 'PostgreSQL',
        'rationale': '',
        'rejected': '',
        'hypothesis': 'serializer corrupts timestamps',
        'fact': 'seed reproduces the flake',
        'provenance': '',
        'evidence': 'perf log was long',
        'transient': '',
        'added': 'team prefers Redis for new services',
    }

def typed_compact(src):
    return {
        'prohibition': src['prohibition'],
        'version': src['version'],
        'migration_id': src['migration_id'],
        'decision': src['decision'],
        'rationale': src['rationale'],
        'rejected': src['rejected'],
        'hypothesis': src['hypothesis'],
        'fact': src['fact'],
        'provenance': '',
        'evidence': 'EVIDENCE-REF perf-log; finding: p99 doubles under batch writes',
        'transient': '',
    }

def typed_plus_prov(src):
    out = typed_compact(src)
    out['provenance'] = src['provenance']
    return out

def oracle_compact(src):
    out = typed_plus_prov(src)
    out['transient'] = src['transient']  # oracle spends budget even on trivia: it knows the probes
    return out

CONDITIONS = {'RAW': (RAW, RAW_TOKENS), 'NAIVE': (naive_compact(SOURCE), 300),
              'TYPED': (typed_compact(SOURCE), 300), 'TYPED+PROV': (typed_plus_prov(SOURCE), 330),
              'ORACLE': (oracle_compact(SOURCE), 360)}
for name, (_, toks) in CONDITIONS.items():
    print(f'{name:10s} {toks} tokens (ratio {toks / SOURCE_TOKENS:.2f})')

## Observation — the survival matrix, computed from probes

Rows are information classes, columns conditions. Cells are computed by probe functions over the representations — never hand-authored. `?` never appears: every probe returns survived or not.

In [ ]:
def probes(rep):
    blob = ' '.join(rep.values())
    return {
        'exact prohibition': PIN_SPANS[0] in blob,
        'identifier': '18.2' in blob and '01947' in blob,
        'decision rationale': 'serialisable transactions' in blob,
        'uncertainty status': 'suspected' in blob,
        'provenance': 'ADR-007' in blob,
        'evidence finding': 'p99 doubles' in blob,
    }

MATRIX = {name: probes(rep) for name, (rep, _) in CONDITIONS.items()}
rows = list(probes(RAW))
print(f"{'class':20s} {' '.join(f'{n:>10s}' for n in CONDITIONS)}")
for r in rows:
    print(f"{r:20s}", ' '.join('[x]' if MATRIX[n][r] else '[ ]' for n in CONDITIONS))
assert MATRIX['RAW']['exact prohibition'] and MATRIX['ORACLE']['provenance']
assert not MATRIX['NAIVE']['exact prohibition']
assert MATRIX['TYPED']['exact prohibition'] and not MATRIX['TYPED']['provenance']
assert MATRIX['TYPED+PROV']['provenance']

## Failure families — omission, mutation, addition, collapse, provenance loss

In [ ]:
naive = CONDITIONS['NAIVE'][0]
print('omission:          rationale survived:', MATRIX['NAIVE']['decision rationale'])
print('mutation:          18.2 ->', repr(naive['version']), '| 01947 ->', repr(naive['migration_id']))
print('addition:          team prefers Redis... in source:', 'team prefers Redis' in ' '.join(SOURCE.values()))
print('epistemic collapse: suspected ->', repr(naive['hypothesis']))
print('provenance loss:   fact survived:', MATRIX['NAIVE']['evidence finding'], '| source survived:', MATRIX['NAIVE']['provenance'])
assert naive['version'] == '18' and SOURCE['version'] == '18.2'
assert 'suspected' not in naive['hypothesis'] and 'suspected' in SOURCE['hypothesis']
assert 'added' in naive and 'added' not in SOURCE
print('Compaction invents as well as omits.')

## Ratio versus fidelity — budget metric beside fidelity metric

In [ ]:
for name, (_, toks) in CONDITIONS.items():
    survived = sum(MATRIX[name].values())
    print(f'{name:10s} ratio={toks / SOURCE_TOKENS:.2f}  classes-survived={survived}/{len(rows)}')
assert sum(MATRIX['NAIVE'].values()) < sum(MATRIX['TYPED'].values())
print('NAIVE and TYPED share ratio 0.25; survival differs. Shorter never means more faithful.')

## Bypass channel and repeated compaction

In [ ]:
for pin in PIN_SPANS:
    assert pin in ' '.join(typed_compact(SOURCE).values()), 'PIN spans bypass semantic transformation'
print('PIN spans byte-identical through typed compaction.')

# Recursive rounds: compact the compaction. Regenerated: rebuild round-3 view from source.
r1, r2, r3 = naive_compact(SOURCE), None, None
r2 = naive_compact({k: v for k, v in r1.items() if k in SOURCE})
r3 = naive_compact({k: v for k, v in r2.items() if k in SOURCE})
regen3 = {'version': '18', 'prohibition': r1['prohibition']}  # rebuilt from retained source, same budget
lineage = [('r1', None), ('r2', 'r1'), ('r3', 'r2'), ('regen3', 'source')]
print('recursive r3 version:', repr(r3.get('version')), '| lineage:', lineage[2])
print('summary-of-summary and new-representation-from-source have different lineage by construction.')

## Try it

1. Give NAIVE the provenance span and re-run the matrix — one cell flips; the prohibition row does not.
2. Change the typed budget to 300 but drop rationale: TYPED loses its distinguishing row at identical ratio.
3. Add a fourth recursive round and confirm lineage length grows while the matrix cannot improve.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(probes(naive_compact(SOURCE)))

## What this demonstrates

- Smaller representations are lossy information-selection operations: quality is what survives by class, not fluency or length.
- Exact, semantic, epistemic, and provenance survival are different requirements; NAIVE fails each independently.
- Compression ratio is a budget metric; survival is the fidelity metric.

## What this does not demonstrate

- That real LLM compactors fail at these rates.
- That typed compaction improves downstream model behaviour.
- That any schema is optimal, or that any fluent summary is good or bad.
- That deterministic fixture loss predicts production summary drift.
- That compression beats pruning or externalisation.

## Connection to the chapter

Full detail and one compact summary form a cliff — full fidelity, then whatever the summariser kept:

> Full detail and one compact summary form a cliff. The next question is whether the same information can exist at several legal levels of fidelity.

That is Chapter 12.